## Download the tables. These tables are already ingested in:
- CATALOG: energy_endava_193
- SCHEMA: default


In [0]:

from pyspark.sql.functions import year, month, dayofmonth, col, to_date, lit

# Read base tables from Delta Lake
nsw_scada = spark.table("energy_endava_193.default.nsw_scada")
nsw_dictionary_mapped = spark.table("energy_endava_193.default.nsw_dictionary_mapped")
nsw_generators_scheduled = spark.table("energy_endava_193.default.nsw_generators_scheduled")
nsw_generators_non_scheduled = spark.table("energy_endava_193.default.nsw_generators_non_scheduled")
nsw_loads = spark.table("energy_endava_193.default.nsw_loads")
nsw_dispatch_demand = spark.table("energy_endava_193.default.nsw_dispatch_demand")
nsw_dispatch_demand_peak_2022 = spark.table("energy_endava_193.default.nsw_dispatch_demand_peak_2022")
nsw_scada_peak_2022 = spark.table("energy_endava_193.default.nsw_scada_peak_2022")

In [0]:
display(nsw_scada_peak_2022)

In [0]:
# Map nsw_scada_peak_2022 with nsw_dictionary_mapped on DUID and add requested columns
nsw_scada_peak_2022 = nsw_scada_peak_2022.join(
    nsw_dictionary_mapped.select(
        "DUID", "DISPATCHTYPE", "SCHEDULE_TYPE", "STARTTYPE", "TECHNOLOGYTYPEDESCRIPTOR"
    ),
    on="DUID",
    how="left"
)
display(nsw_scada_peak_2022)

In [0]:
display(nsw_dispatch_demand_peak_2022)

In [0]:
from pyspark.sql.functions import when, isnull, row_number
from pyspark.sql.window import Window

# Deduplicate nsw_dictionary_mapped by keeping only the most recent entry per DUID
window_spec = Window.partitionBy("DUID").orderBy(col("START_DATE").desc())
nsw_dictionary_dedup = nsw_dictionary_mapped.withColumn("row_num", row_number().over(window_spec)).filter(col("row_num") == 1).drop("row_num")

# Recreate the join from Cell 4 with deduplicated dictionary
nsw_scada_enriched = spark.table("energy_endava_193.default.nsw_scada_peak_2022").join(
    nsw_dictionary_dedup.select(
        "DUID", "DISPATCHTYPE", "SCHEDULE_TYPE", "STARTTYPE", "TECHNOLOGYTYPEDESCRIPTOR"
    ),
    on="DUID",
    how="left"
)

# Add DUMMYCOLUMN
nsw_scada_dummy = nsw_scada_enriched.withColumn(
    "DUMMYCOLUMN",
    when(isnull(col("TECHNOLOGYTYPEDESCRIPTOR")), lit("NULL")).otherwise(lit("NOT NULL"))
)

# Count rows for each unique combination
nsw_scada_dummy_grouped = nsw_scada_dummy.groupBy(
    "SCHEDULE_TYPE", "STARTTYPE", "DISPATCHTYPE", "DUMMYCOLUMN"
).count()

display(nsw_scada_dummy_grouped)